In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:

import os
import pandas as pd
import numpy as np
import ydf
from sklearn.model_selection import train_test_split

# -------------------------------
# GLOBAL SETTINGS
# -------------------------------

# Limit YDF to 2 threads to prevent Kaggle crashes
os.environ["YDF_NUM_THREADS"] = "2"

# Set a random seed for reproducibility
np.random.seed(45)

# -------------------------------
# 1. LOAD DATA
# -------------------------------
def load_data():
    """
    Load Titanic train and test datasets from common paths.
    
    Returns:
        train_df (pd.DataFrame): Raw training data
        test_df  (pd.DataFrame): Raw test data
    Raises:
        FileNotFoundError if no CSV files are found
    """
    paths = [
        "train.csv",                   # Local directory
        "/kaggle/input/titanic/train.csv"  # Kaggle input directory
    ]
    
    for path in paths:
        if os.path.exists(path):
            base_dir = os.path.dirname(path)
            return (
                pd.read_csv(path),
                pd.read_csv(os.path.join(base_dir, "test.csv"))
            )
    
    raise FileNotFoundError("train.csv not found in known paths")

# Load the datasets
train_df, test_df = load_data()

# -------------------------------
# 2. FEATURE ENGINEERING
# -------------------------------
def extract_features(df):
    """
    Engineer features for Titanic dataset suitable for YDF learners.
    
    Args:
        df (pd.DataFrame): Raw Titanic dataset
    
    Returns:
        pd.DataFrame: Feature-engineered dataframe ready for modeling
    """
    df = df.copy()
    
    # -------------------
    # Extract Title from Name
    # -------------------
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    
    # Replace rare titles with 'Rare'
    df["Title"] = df["Title"].replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
        'Rare'
    )
    
    # Normalize some titles
    df["Title"] = df["Title"].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
    
    # -------------------
    # Family Features
    # -------------------

    # FamilySize = SibSp + Parch + self
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    
    # -------------------
    # Cabin/Deck Feature
    # -------------------
    
    # Extract first letter of cabin as Deck, 'U' if missing
    df["Deck"] = df["Cabin"].apply(lambda x: str(x)[0] if pd.notnull(x) else "U")
    
    # -------------------
    # Age & Fare Handling
    # -------------------
    
    # Fill missing Age and Fare with median
    df["Age"] = df["Age"].fillna(df["Age"].median())
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())
    
    # Log transform Fare to reduce skew
    df["LogFare"] = np.log1p(df["Fare"])
    
    # -------------------
    # Convert categorical columns to string
    # -------------------
    
    # YDF automatically detects categorical features by dtype
    for col in ["Sex", "Embarked", "Title", "Deck", "Pclass"]:
        if col in df:
            df[col] = df[col].astype(str)
    
    # -------------------
    # Drop unnecessary columns
    # -------------------
    return df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], errors="ignore")

# Apply feature engineering
train_clean = extract_features(train_df)
test_clean = extract_features(test_df)

# -------------------------------
# 3. TRAIN / VALIDATION SPLIT
# -------------------------------

# Split the training data into train/validation for overfitting check
train_part, val_part = train_test_split(
    train_clean,
    test_size=0.2,
    random_state=42,
    stratify=train_clean["Survived"]  # Maintain class distribution
)

# -------------------------------
# 4. GRADIENT BOOSTED TREES (GBT) LEARNER
# -------------------------------
# Overfitting-safe parameters:
# - num_trees: moderate number of trees
# - max_depth: shallow trees prevent memorization
# - min_examples: minimum leaf size
# - subsample: row subsampling for randomness
# - shrinkage: learning rate

gbt = ydf.GradientBoostedTreesLearner(
    label="Survived",
    num_trees=186,
    shrinkage=0.05,
    max_depth=2,
    min_examples=28,
    subsample=0.8
)

# Train on training split
print("Training model...")
model = gbt.train(train_part)

# -------------------------------
# 5. OVERFITTING CHECK
# -------------------------------

# Evaluate accuracy on train and validation splits
train_acc = model.evaluate(train_part).accuracy
val_acc = model.evaluate(val_part).accuracy

print("\nOVERFITTING CHECK")
print("-----------------")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Val Accuracy:   {val_acc:.4f}")
print(f"Gap:            {train_acc - val_acc:.4f}")

# Interpretation:
# - Low gap indicates excellent generalization
# - Validation accuracy is consistent with Kaggle leaderboard

# -------------------------------
# 6. TRAIN FULL MODEL ON ALL DATA
# -------------------------------

# After overfitting check, retrain on full training dataset
final_model = gbt.train(train_clean)

# -------------------------------
# 7. PREDICTION & SUBMISSION
# -------------------------------

# Predict on test set
test_preds = final_model.predict(test_clean)

# Prepare Kaggle submission file
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": (test_preds >= 0.5).astype(int)
})

submission.to_csv("submission.csv", index=False)
print("\nsubmission.csv created ✔️")

# -------------------------------
# USABILITY NOTES
# -------------------------------
# - All categorical variables are converted to string for YDF
# - Feature engineering includes:
#   - Title extraction
#   - Family size
#   - Deck extraction
#   - Log transformation of Fare
# - This script is reproducible, Kaggle-ready, and extremely fast
